# LULC

This notebook does 3 things:
1. Generate training data for LULC classification. It uses 3 existing LULC products.
2. Filter training data per class using kmeans clustering to exclude outliers.
3. Train a random forest model, and then predict LULC.

### Steps: 
1. Training data:
For each site extract training data.
  -> Find product agreement. mask to gadm 100m buffer.
  -> Add geomad, indices, elevation etc.
  -> Write csv per site of training data.
do this in a notebook. don't commit training data.

1a. Filter outliers per class.

2. Train model:
Try one model for all sites. (append all CSVs)
export model dump (python pickle?)
in future we may need to make different models for different regions and year ranges.
train the model using the geomad of the year of the input products.

3. Predict:
per grid tile:
  per year:
    load geomad/indices/elevation etc.
    make using get_gadm (buffered 100m)
    predict
This is as a command. 

### Load packages


In [ ]:
# Not sure if this is needed.
# Reload functions during development
%load_ext autoreload
%autoreload 2

In [ ]:
%matplotlib inline

# Scientific core
import numpy as np
import pandas as pd

# Visualisation
import matplotlib.gridspec as gridspec
from matplotlib import pyplot as plt
from matplotlib.colors import BoundaryNorm, ListedColormap
from matplotlib.patches import Patch

# Geospatial
import xarray as xr
from ipyleaflet import basemaps
from odc.geo.xr import assign_crs
from odc.stac import load
from planetary_computer import sign_url
from pystac.client import Client
import rioxarray # noqa: F401
import geopandas as gpd
from antimeridian import fix_multi_polygon, fix_polygon

# Machine learning
from scipy.ndimage import minimum_filter
from scipy.spatial.distance import cdist
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import ConfusionMatrixDisplay, classification_report, confusion_matrix, silhouette_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Local
from ldn.grids import get_gadm
from ldn.random_sampling import random_sampling
from ldn.typology import classes, colors, world_cover_map, cci_lc_map, io_map, lvl1
from ldn.utils import get_geomad_dem_indices
from notebooks.src.Compare_LULC_func import standardise_class

In [ ]:
# Parameters
datetime = '2020-06' # TODO: Just use 2020?
datetime_year = datetime.split("-")[0]

wgs84 = 'EPSG:4326' # WGS84
class_attr='lulc'
buffer_m  = 100
# Use Planetary Computer STAC API
catalog = Client.open("https://planetarycomputer.microsoft.com/api/stac/v1/")


# For Singapore:
# country_of_interest = {"Singapore": "SGP"}
# analysis_crs = 'EPSG:6933' # Equal Earth for global analysis
# stac_geoparquet = "https://s3.us-west-2.amazonaws.com/data.ldn.auspatious.com/ci_ls_geomad/0-0-2/ci_ls_geomad.parquet" # Non-Pacific
split_at_antimeridian = False # No chance that this country will intersect the antimeridian.
# For Fiji:
country_of_interest = {"Fiji": "FJI"}
analysis_crs = 'EPSG:3832' # PDC Mercator for antimeridian avoiding
stac_geoparquet = "https://s3.us-west-2.amazonaws.com/data.ldn.auspatious.com/ausp_ls_geomad/0-0-2/ausp_ls_geomad.parquet" # Pacific
split_at_antimeridian = True  # No chance that this country will intersect the prime meridian.

# TODO: For Fiji (and all Pacific countries to be safe against the antimeridian), I could do this whole process for east and west hemispheres and then merge the training data points at the end.

lulc_class_raster = f'lulc_agree_{list(country_of_interest.keys())[0]}.tif'

In [ ]:
# country_wgs84 = get_gadm(countries=country_of_interest, overwrite=True)
# country_wgs84_single = country_wgs84.explode(index_parts=False).reset_index(drop=True) # Split to singlepart.

In [ ]:
# Make buffered country polygon to clip with.
def fix_multi_or_polygon(geom):
    print(f"Original geometry type: {geom.geom_type}")
    return fix_multi_polygon(geom) if geom.geom_type == "MultiPolygon" else fix_polygon(geom)

country_wgs84_buffered = (
    get_gadm(countries=country_of_interest, overwrite=True)
    .to_crs(analysis_crs)
    .buffer(buffer_m)
    .to_crs(wgs84)
    .apply(fix_multi_or_polygon)
)
country_wgs84_buffered = gpd.GeoDataFrame(geometry=country_wgs84_buffered, crs=wgs84)
country_wgs84_buffered.explore()

In [ ]:
from shapely.geometry import box

regions = []

if split_at_antimeridian:
    print("2 zones are east and west hemispheres.")
    east = gpd.clip(country_wgs84_buffered, box(0, -90, 180, 90))
    west = gpd.clip(country_wgs84_buffered, box(-180, -90, 0, 90))
    regions = [east, west]
else:
    print("1 zone covering the whole globe and country. No splitting at the antimeridian needed.")
    regions = [country_wgs84_buffered]

print(f"Number of regions: {len(regions)}")
print((regions[0].geometry.total_bounds))
print((regions[1].geometry.total_bounds))
# regions_gdf = gpd.GeoDataFrame(pd.concat(regions, ignore_index=True), crs=wgs84)
# regions[0].explore()
regions[1].explore()

In [ ]:
# I am no longer doing this since changing to using 2 regions instead of 1 region that straddles the antimeridian.

# # Need to do this our selves. It makes nice bboxes that don't straddle the antimeridian, which causes problems for STAC search and for loading data with ODC.
# # Otherwise the bbox of Fiji spans the circumference of the world and loads way too much data.
# # Can't use fix_polygon because that only splits geometries along the antimeridian, which is already done and doesn't fix this issue.

# def split_antimeridian_bbox(gdf: GeoDataFrame) -> GeoDataFrame:
#     """
#     For AOIs straddling the antimeridian, returns a GeoDataFrame with two bboxes:
#     one covering the eastern hemisphere parts, one covering the western hemisphere parts.
#     For AOIs not straddling the antimeridian, just return a GeoDataFrame with one bbox.
#     """
#     minx, miny, maxx, maxy = gdf.total_bounds
#     print(f"Original bounds: {minx}, {miny}, {maxx}, {maxy}")

#     if minx > -179.9 and maxx < 179.9:
#         print("AOI does not straddle the antimeridian.")
#         return GeoDataFrame(geometry=[BoundingBox(minx, miny, maxx, maxy).polygon], crs="EPSG:4326")

#     print("AOI straddles the antimeridian. Splitting into east/west hemisphere bboxes.")
#     east_parts, west_parts = [], []

#     for geom in gdf.geometry:
#         gx_min, gy_min, gx_max, gy_max = geom.bounds
#         if gx_min >= 0:
#             east_parts.append((gx_min, gy_min, gx_max, gy_max)) # Completely in eastern hemisphere
#         elif gx_max <= 0:
#             west_parts.append((gx_min, gy_min, gx_max, gy_max)) # Completely in western hemisphere
#         else:
#             raise ValueError("Country bbox should not straddle the antimeridian and the prime meridian.")


#     def union_bbox(parts) -> BoundingBox:
#         return BoundingBox(
#             max(min(p[0] for p in parts), -179.999), # Clamp to -180 because that causes problems for STAC search and ODC loading.
#             min(p[1] for p in parts),
#             min(max(p[2] for p in parts), 179.999), # Clamp to 180 because that causes problems for STAC search and ODC loading.
#             max(p[3] for p in parts),
#         )

#     bboxes = []
#     if east_parts:
#         bboxes.append(union_bbox(east_parts).polygon)
#     if west_parts:
#         bboxes.append(union_bbox(west_parts).polygon)

#     result = GeoDataFrame(geometry=bboxes, crs="EPSG:4326")
#     print(result.bounds)
#     return result


# print(f"Country has {len(country_wgs84_single['geometry'])} polygons")

# country_bounds_wgs84 = split_antimeridian_bbox(country_wgs84_single)

# print(country_bounds_wgs84)
# # print(country_buffered.bounds)
# country_bounds_wgs84.explore()

In [ ]:
print(len(regions[0].geometry))

# This takes X mins for Fiji.
region_geomad_dem = []

for region in regions:
    print("Processing region with bounds:", region.geometry.total_bounds)
    region_geomad_dem.append(get_geomad_dem_indices(region, stac_geoparquet, datetime_year, catalog=catalog))

# Singapore and Fiji only have 2 GeoMAD tiles for 2020 as of 2026-03-05. GeoMAD will be fully processed in future.

# Write all bands
# region_geomad_dem[0].to_array(dim="band").odc.write_cog("fiji_test_region_0.tif", overwrite=True)
# Can just write a single band to check it looks correct.
region_geomad_dem[0]['red'].odc.write_cog("fiji_test_region_0.tif", overwrite=True)

In [ ]:
# This cell is just to check the data visually.
for column in country_geomad_dem.data_vars:
    # print(f"{column}: {country_geomad_dem[column].shape}, {country_geomad_dem[column].dtype}")
    print(f"Column: {column}, min: {country_geomad_dem[column].min().values}, max: {country_geomad_dem[column].max().values}, dtype: {country_geomad_dem[column].dtype}")

# test_attr = "slope"
test_attr = "elevation"
# test_attr = "aspect"
# test_attr = "ndvi"
# test_attr = "red"
test_attr_colormap = {"elevation": "terrain", "aspect": "viridis", "slope": "plasma", "ndvi": "YlGn", "red": "Reds"}[test_attr]

country_geomad_dem[test_attr].odc.explore(
    cmap=test_attr_colormap,
    vmin=country_geomad_dem[test_attr].min().values,
    vmax=country_geomad_dem[test_attr].max().values,
)

# For some reason for Fiji the data looks really low res, but it is actually 10m. Maybe .explore() downsamples because the extent is huge?

## Use 3 LU/LC products. Find agreement between them.

In [ ]:
# Steps:
# 1. Load the highest res product (there are 2 at 10m.)
# 2. Load the others to be like the 1st. Use nearest resampling to avoid creating new classes that may not exist in the original data.
# 3. Find agreement where 2 out of 3 of the products match exactly. CCI is probably going to agree the least due to its different resolution. Neighbours also agree.

# TODO: Experiment with data quality bands - some LULC datasets have quality information which could be used as additional filters. Use filter in search/load.

# Handle antimeridian split, hemisphere split bboxes for countries like Fiji.
# A generic function to load a product given the boundary
def load_lulc_data(product: str, like: xr.Dataset | None) -> xr.Dataset:
    # Search across each bbox polygon (handles antimeridian splits)
    stac_docs = {}
    for bbox_polygon in country_bounds_wgs84.geometry:
        print(f"Searching {product} with bbox: {bbox_polygon.bounds}")
        for item in catalog.search(
            collections=[product],
            bbox=list(bbox_polygon.bounds),
            datetime=datetime,
        ).items():
            stac_docs[item.id] = item

    items = list(stac_docs.values())
    print(f"Found {len(items)} unique STAC items for product {product}")

    if like is not None:
        # Aligned to first product grid — like= handles extent correctly
        ds = load(
            items,
            like=like,
            chunks={},
            patch_url=sign_url,
            resampling="nearest",
        )
    else:
        # First product — load each bbox side separately to avoid antimeridian world-spanning extent
        # TODO: This may have some uneeded complexity.
        parts = []
        for bbox_polygon in country_bounds_wgs84.geometry:
            side_items = [item for item in items if any(
                bbox_polygon.bounds[0] <= float(item.bbox[0]) <= bbox_polygon.bounds[2] or
                bbox_polygon.bounds[0] <= float(item.bbox[2]) <= bbox_polygon.bounds[2]
                for _ in [None]
            )]
            if not side_items:
                continue
            print(f"Loading {len(side_items)} items for bbox {bbox_polygon.bounds}")
            part = load(
                side_items,
                bbox=list(bbox_polygon.bounds),
                crs=wgs84,
                # resolution=resolution, # Load native resolution. WGS84 is in degrees so using resolution in metres breaks loading.
                chunks={},
                patch_url=sign_url,
                resampling="nearest",
            )
            # Clip lazily BEFORE .load() to avoid materialising the full tile
            part = part.isel(time=0)
            part = part.rio.set_spatial_dims(x_dim="longitude", y_dim="latitude")
            part = part.rio.write_crs(wgs84, inplace=True)
            part = part.rio.clip(
                country_wgs84_buffered.geometry.values,
                crs=wgs84,
                all_touched=True,
                from_disk=True, # clips lazily without loading full array
            )
            part = part.load() # only load the clipped region
            parts.append(part)

        ds = xr.merge(parts) if len(parts) > 1 else parts[0]

    print(f"Product {product} has variables: {ds.data_vars}")
    return ds

In [ ]:
LULC_PRODUCTS = [
    ("esa-worldcover",      "map",        "esa_wc", world_cover_map), # 10m
    ("esa-cci-lc",          "lccs_class", "esa_cci",     cci_lc_map), # 300m
    ("io-lulc-annual-v02",  "data",       "io",        io_map), # 10m
]

def load_and_prepare(product: str, native_band: str, output_band: str, class_map: dict, like: xr.Dataset | None) -> xr.Dataset:
    ds = load_lulc_data(product, like=like)
    ds[output_band] = ds[native_band]
    ds = ds.drop_vars([v for v in ds.data_vars if v != output_band]) # Remove all vars but the LC one.
    ds = ds.squeeze().load() # Removes any singleton dimension and then load the full data
    ds = ds.rio.set_spatial_dims(x_dim="longitude", y_dim="latitude")
    print(ds.dims)           # confirm dim names
    print(ds.rio.crs)        # confirm CRS is set
    print(ds.rio.bounds())   # confirm spatial extent makes sense for Fiji
    print(country_wgs84_buffered.total_bounds)  # confirm overlap
    ds = ds.rio.clip(country_wgs84_buffered.geometry.values, crs=wgs84) # Clip to buffered AOI. ODC STAC Load intersects param doesn't clip.
    ds[output_band] = standardise_class(ds[output_band], class_map)
    ds[output_band].odc.write_cog(f"{output_band}_standardised.tif", overwrite=True)
    return ds

# Load worldcover first (highest native res), then align others to it
wc_10m,  cci_10m,  io_10m  = None, None, None

for i, (product, native_band, output_band, class_map) in enumerate(LULC_PRODUCTS):
    like = wc_10m if i > 0 else None
    ds = load_and_prepare(product, native_band, output_band, class_map, like)
    if i == 0:
        wc_10m  = ds
    elif i == 1:
        cci_10m = ds
    else:
        io_10m  = ds

In [ ]:
wc_10m["esa_wc"].odc.explore()
# wc_10m["esa_wc"]

In [ ]:
cci_10m["esa_cci"].odc.explore()
# cci_10m["esa_cci"]

In [ ]:
io_10m["io"].odc.explore()

In [ ]:
# Standardise the classes to match the typology
cci_10m['esa_cci'] = standardise_class(cci_10m['esa_cci'], cci_lc_map)
wc_10m['esa_wc'] = standardise_class(wc_10m['esa_wc'], world_cover_map)
io_10m['io'] = standardise_class(io_10m['io'], io_map)

# Write mapped products as tiff.
cci_10m["esa_cci"].odc.write_cog("cci_10m_standardised.tif", overwrite=True)
wc_10m["esa_wc"].odc.write_cog("wc_10m_standardised.tif", overwrite=True)
io_10m["io"].odc.write_cog("io_10m_standardised.tif", overwrite=True)

wc_10m

In [ ]:
# Print class distributions before agreement analysis.
for name, ds, band in [
    ("wc", wc_10m, 'esa_wc'),
    ("cci",        cci_10m, 'esa_cci'),
    ("io",         io_10m,  'io'),
]:
    vals, counts = np.unique(ds[band].values, return_counts=True)
    print(f"\n=== {name} ===")
    print(pd.DataFrame({"class": vals, "count": counts}))

# CCI has no Other (7) class in Singapore.

# Create layer where all 2 out of 3 land cover products agree.
Each pixel and its 8 neighbours must have agreement for 2 out of the 3 products.

In [ ]:
# Valid pixels — all three products have data
valid = (wc_10m['esa_wc'] > 0) & (cci_10m['esa_cci'] > 0) & (io_10m['io'] > 0)

# Count pairwise agreements at each pixel (0, 1, 2, or 3 pairs agree)
wc  = wc_10m['esa_wc']
cci = cci_10m['esa_cci']
io  = io_10m['io']

pair_agree_count = (
    (wc == cci).astype("int8") +
    (cci == io).astype("int8") +
    (wc == io).astype("int8")
)

# At least 2 of 3 products agree — take the majority class
majority_class = xr.where(wc == cci, wc, io)  # if wc==cci, that's the majority; else io agrees with one of them
two_of_three = (pair_agree_count >= 2) & valid

# All 8 neighbours must also have 2/3 agreement
neighbour_mask = xr.DataArray(
    minimum_filter(two_of_three.values.astype("float32"), size=3, mode="constant", cval=0) == 1,
    coords=two_of_three.coords, dims=two_of_three.dims,
)

agreed_class = majority_class.where(neighbour_mask & two_of_three, other=0).rename(class_attr)

# Diagnostics
label_map = {c["value"]: c["label"] for c in lvl1.values()}
_classes, counts = np.unique(agreed_class.values, return_counts=True)
counts_df = pd.DataFrame({"class": _classes, "pixel_count": counts})
counts_df["label"] = counts_df["class"].map(label_map)
print(counts_df)

In [ ]:
# Write agreeing pixels as tiff.
agreed_class.odc.write_cog(lulc_class_raster, overwrite=True)

classification_map = agreed_class
classification_map

# Visualise the agreeing classes from 3 products

In [ ]:
fig, axes = plt.subplots(1,1)

# Plot classification map
unique_values=np.unique(classification_map)
cmap=ListedColormap([colors[k] for k in unique_values])
norm = BoundaryNorm(list(unique_values)+[np.max(unique_values)+1], cmap.N)
classification_map.plot.imshow(ax=axes, 
                   cmap=cmap,
                   norm=norm,
                   add_labels=True, 
                   add_colorbar=False,
                   interpolation='none')
# add colour legend
patches_list=[Patch(facecolor=colour) for colour in colors.values()]
axes.legend(patches_list, list(classes.keys()),loc='upper center', ncol =4, bbox_to_anchor=(0.5, -0.1))

## Generate random training samples
We generate some randomly distributed samples for each class from the clipped classification map using the `random_sampling` function. This function takes in a few parameters:  
* `da`: a classified map in the format of 2-dimensional xarray.DataArray
* `n`: total number of points to sample.
* `min_sample_n`: Minimum number of samples to generate per class if proportional number is smaller than this. **Note that the resultant number of samples may be higher than the set `n` due to setting of this minimum number of samples.** 
* `sampling`: the sampling strategy, e.g. 'stratified_random' where each class has a number of points proportional to its relative area, 'equal_stratified_random' where each class has the same number of points, or 'manual' which allows you to define number of samples for each class.
* `out_fname`: a filepath name for the function to export a shapefile/geojson of the sampling points into a file. You can set this to `None` if you don't need to output the file.
* `class_attr`: This is the column name of output dataframe that contains the integer class values on the classified map. 
* `drop_value`: pixel value on the classification map to be excluded from sampling.

The output of the function is a geopandas dataframe of randomly distributed points containing a column `class_attr` identifying class values. 

Here we extract around 1000 training points in total and export the points in a geojson file for use in the rest of workflow. Here we use the stratified sampling method by setting 'equal_stratified_random', but also set the minimum number of samples as 75 to avoid missing samples for some minor classes. 

As mentioned earlier we don't want the abandoned classes to be included in the samples we set drop_value as 0 before implementing the function:

In [ ]:
# Reproject to WGS84 so sample points are in lat/lon
# TODO: Don't reproject. It should be loaded correctly in the first place. Maybe all product data should be loaded in WGS84 to begin with? Or I can tweak random_sampling to accept other CRSs.
# TODO: Reprojecting is a red flag. It can cause a lot of issues. At least use nearest neighbour to avoid creating new classes that may not exist in the original data.
# classification_map_4326 = classification_map.rio.reproject(output_crs, resampling=rasterio.enums.Resampling.nearest)
classification_map_4326.head()

In [ ]:
n=2100
min_sample_n=300
drop_value=0
gpd_random_samples=random_sampling(da=classification_map_4326,n=n,sampling='stratified_random',
                                   min_sample_n=min_sample_n,out_fname=None,class_attr=class_attr,drop_value=drop_value)

In [ ]:
print(f"CRS: {gpd_random_samples.crs}")
gpd_random_samples.head()

## Visualise the training data by class

In [ ]:
gpd_random_samples.explore(
    column=class_attr,
    categorical=True,
    categories=(present_classes := sorted(gpd_random_samples[class_attr].unique())),
    cmap=[colors[c] for c in present_classes],
    legend=True,
    style_kwds={"radius": 6, "fillOpacity": 0.8, "weight": 0.5}
)

## Extract Geomedian/GeoMAD values at training data point locations.

For now we run and load individual tiles and mosaic them because we have not run all tiles yet.

Once we have finalised the GeoMAD data, we will set up a GeoParquet STAC index for it. Then querying it by bbox will be simple and replace the more complicated process implemented here.

In [ ]:
band_names_with_indices = [v for v in country_geomad_dem.data_vars]
print(band_names_with_indices)

# Extract Geomedian and GeoMAD elevation, indices values at each sample point location

# Reproject sample points to match native CRS of geomad before sampling.
gpd_random_samples_analysis = gpd_random_samples.to_crs(analysis_crs)

# Build point-wise x/y arrays with the SAME 'points' index
points_idx = np.arange(len(gpd_random_samples_analysis))
xs = xr.DataArray(gpd_random_samples_analysis.geometry.x.values, dims="points", coords={"points": points_idx})
ys = xr.DataArray(gpd_random_samples_analysis.geometry.y.values, dims="points", coords={"points": points_idx})
# Point-wise nearest sampling (one sampled pixel per input point)
sampled = country_geomad_dem[band_names_with_indices].sel(x=xs, y=ys, method="nearest")

# Convert to dataframe indexed by points; keep exactly one row per point
sampled_df = sampled.to_dataframe().reset_index()
sampled_df = sampled_df.sort_values("points").reset_index(drop=True)

# Merge back to training points (original CRS)
training_data = gpd_random_samples.reset_index(drop=True).copy()
training_data = pd.concat([training_data, sampled_df[band_names_with_indices]], axis=1)

print(f"Training data shape: {training_data.shape}")
print("Unique values per band:")
print(training_data[band_names_with_indices].nunique())
training_data.head()

## Visualise GeoMedian red values per sample point

In [ ]:
# Visualise red values using the actual data range
training_data.explore(
    column="red",
    cmap="Reds",
    k=7,
    scheme="natural_breaks",
    legend=True,
    style_kwds={"radius": 4, "fillOpacity": 0.8, "weight": 0.5},
)

## Explore elevation data on training data points

In [ ]:
training_data.explore(
    column="elevation",
    cmap="Blues",
    scheme="natural_breaks",
    k=7,
    legend=True,
    style_kwds={"radius": 4, "fillOpacity": 0.8, "weight": 0.5},
)

## Explore NDVI data on training data points

In [ ]:
training_data.explore(
    column="ndvi",
    cmap="RdYlGn",
    scheme="quantiles",
    k=7,
    legend=True,
    style_kwds={"radius": 4, "fillOpacity": 0.8, "weight": 0.5},
)

# Section 2: Filter training data by K-Means clustering
Goal is to reduce confusion. Make training point classes clean/accurate/spectrally separable.

In [ ]:
# 1. Prep feature matrix ────────────────────────────────────────────────────
exclude_cols = ['lulc', 'geometry']
feature_cols = [c for c in training_data.columns if c not in exclude_cols]

training_data['outlier'] = False
training_data.loc[training_data[feature_cols].isna().any(axis=1), 'outlier'] = True

nan_rows = training_data['outlier'].sum()
print(f"Rows with NaNs (auto-flagged): {nan_rows}")

valid_mask = ~training_data[feature_cols].isna().any(axis=1)
valid_idx  = training_data.index[valid_mask]

X = training_data.loc[valid_idx, feature_cols].values
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# 2. Visualisation helpers ──────────────────────────────────────────────────
CLUSTER_CMAP = 'tab10'

def plot_pca(ax, Xc, labels, centers, outlier_mask, class_name, best_score):
    """PCA scatter — inliers coloured by cluster, outliers as red crosses."""
    pca      = PCA(n_components=2)
    X_2d     = pca.fit_transform(Xc)
    c_2d     = pca.transform(centers)
    var      = pca.explained_variance_ratio_
    k        = len(np.unique(labels))
    inliers  = ~outlier_mask

    ax.scatter(
        X_2d[inliers, 0], X_2d[inliers, 1],
        c=labels[inliers], cmap=CLUSTER_CMAP, vmin=0, vmax=max(k - 1, 1),
        alpha=0.55, s=25, linewidths=0, label='inlier'
    )
    if outlier_mask.any():
        ax.scatter(
            X_2d[outlier_mask, 0], X_2d[outlier_mask, 1],
            c='red', marker='x', s=50, linewidths=1.2, label='outlier'
        )
    ax.scatter(
        c_2d[:, 0], c_2d[:, 1],
        c='black', marker='*', s=180, zorder=5, label='centroid'
    )
    ax.set(
        xlabel=f'PC1 ({var[0]:.1%})',
        ylabel=f'PC2 ({var[1]:.1%})',
        title=f'PCA  |  k={k}  sil={best_score:.3f}'
    )
    ax.legend(fontsize=7, markerscale=0.9)


def plot_heatmap(ax, centers, feature_cols, outlier_count, n):
    """Centroid heatmap — rows = clusters, cols = features."""
    im = ax.imshow(centers, aspect='auto', cmap='RdBu_r')
    ax.set_xticks(range(len(feature_cols)))
    ax.set_xticklabels(feature_cols, rotation=45, ha='right', fontsize=7)
    ax.set_yticks(range(len(centers)))
    ax.set_yticklabels([f'C{i}' for i in range(len(centers))], fontsize=8)
    plt.colorbar(im, ax=ax, label='Scaled value', pad=0.02)

    # Annotate each cell with the scaled value
    for r in range(centers.shape[0]):
        for c in range(centers.shape[1]):
            ax.text(c, r, f'{centers[r, c]:.1f}',
                    ha='center', va='center', fontsize=6,
                    color='white' if abs(centers[r, c]) > 1.2 else 'black')

    ax.set_title(f'Centroids  |  outliers={outlier_count} ({100*outlier_count/n:.1f}%)')


# 3. Per-class clustering + outlier flagging + visualisation ────────────────
threshold = 2.0   # ← tune: lower = more aggressive flagging

_classes = training_data.loc[valid_idx, 'lulc'].unique()

for cls in _classes:
    mask = (training_data.loc[valid_idx, 'lulc'] == cls).values
    idx  = valid_idx[mask]
    Xc   = X_scaled[mask]
    n    = len(Xc)

    if n < 6:
        continue
    k_max = min(5, n // 5)
    if k_max < 2:
        continue

    # Optimal k via silhouette ──────────────────────────────────────────
    best_k, best_score = 2, -1
    for k in range(2, k_max + 1):
        km     = KMeans(n_clusters=k, random_state=42, n_init=10)
        labels = km.fit_predict(Xc)
        score  = silhouette_score(Xc, labels)
        if score > best_score:
            best_k, best_score = k, score

    km_final = KMeans(n_clusters=best_k, random_state=42, n_init=10).fit(Xc)
    labels   = km_final.labels_
    centers  = km_final.cluster_centers_

    # Outlier flagging ──────────────────────────────────────────────────
    outlier_mask = np.zeros(n, dtype=bool)
    for cluster_id in range(best_k):
        in_cluster = labels == cluster_id
        pts        = Xc[in_cluster]
        dists      = cdist(pts, centers[cluster_id].reshape(1, -1)).flatten()
        median_d   = np.median(dists)
        if median_d == 0:
            continue
        flagged = dists > threshold * median_d
        outlier_mask[np.where(in_cluster)[0][flagged]] = True

    training_data.loc[idx[outlier_mask], 'outlier'] = True

    print(f"  {str(cls):30s} | n={n:4d} | k={best_k} | sil={best_score:.3f} "
          f"| outliers={outlier_mask.sum():4d} ({100*outlier_mask.mean():.1f}%)")

    # Figure: PCA (left) + heatmap (right) ─────────────────────────────
    fig = plt.figure(figsize=(14, 4.5))
    fig.suptitle(f'Class: {cls}  (n={n})', fontsize=12, fontweight='bold', y=1.01)

    gs  = gridspec.GridSpec(1, 2, width_ratios=[1, 1.6], figure=fig)
    ax_pca  = fig.add_subplot(gs[0])
    ax_heat = fig.add_subplot(gs[1])

    plot_pca(ax_pca, Xc, labels, centers, outlier_mask, cls, best_score)
    plot_heatmap(ax_heat, centers, feature_cols, outlier_mask.sum(), n)

    plt.tight_layout()
    plt.show()

# 4. Summary ────────────────────────────────────────────────────────────────
print(f"\nTotal points : {len(training_data):,}")
print(f"Clean        : {(~training_data['outlier']).sum():,}")
print(f"Outliers     : {training_data['outlier'].sum():,}")
print(f"\nOutliers per class:")
print(training_data.groupby('lulc')['outlier'].mean().mul(100).round(1).astype(str) + '%')
training_data = training_data[~training_data['outlier']]

In [ ]:
training_data.drop(columns=["time", 'spatial_ref', 'count'], inplace=True)

country_of_interest_name = list(country_of_interest.keys())[0]
out_fname = f'training_data_{country_of_interest_name}'

# Write the final training data
training_data.to_file(f"{out_fname}.geojson", driver="GeoJSON", index=False)
training_data.to_csv(f"{out_fname}.csv", index=False)

print(f"Saved training data as geojson and CSV to {out_fname}")

# Section 3: Train and Predict

In [ ]:
# Find and load training data for all sites
from glob import glob

training_data_files = glob("training_data_*.csv")

print(f"Found {len(training_data_files)} files")

training_data = pd.concat([pd.read_csv(f) for f in training_data_files], ignore_index=True)

In [ ]:
print(training_data.columns)

In [ ]:
training_data.drop(columns=['outlier'], inplace=True)
# training_data.drop(columns=['outlier', 'geometry'], inplace=True)


# Split 70/30 into train/test. Splits the classes into train/test in a representative way.
train_gdf, test_gdf = train_test_split(training_data, test_size=0.3, stratify=training_data[class_attr], random_state=42)

print(f"Training set class distribution:\n{train_gdf[class_attr].value_counts()}")
print(f"Test set class distribution:\n{test_gdf[class_attr].value_counts()}")
print(train_gdf)

## Create a classifier and fit a model

_classes = train_gdf[class_attr].values.astype(int)
print(f"Classes: {np.unique(_classes)}")

# Observations — drop class and any remaining non-numeric columns
feature_cols = [c for c in train_gdf.columns
                if c != class_attr
                and pd.api.types.is_numeric_dtype(train_gdf[c])]
observations = train_gdf[feature_cols].values

# Create and fit
classifier = RandomForestClassifier(class_weight='balanced')
model = classifier.fit(observations, _classes)
# Define features and target

feature_cols = [c for c in train_gdf.columns if c != class_attr]

X_train = train_gdf[feature_cols].values
y_train = train_gdf[class_attr].values
X_test = test_gdf[feature_cols].values
y_test = test_gdf[class_attr].values

classifier = RandomForestClassifier(n_estimators=500, class_weight="balanced", random_state=42)
model = classifier.fit(X_train, y_train)
y_pred = model.predict(X_test)

# Feature importance — drop noisy features
importances = pd.Series(model.feature_importances_, index=feature_cols).sort_values(ascending=False)
print("Feature importances:")
print(importances)
# Feature importance is probably the most useful next step — it'll tell you which bands are actually helping and which are adding noise.

present = np.unique(np.concatenate([y_test, y_pred]))
target_names = [k for k, v in sorted(classes.items(), key=lambda x: x[1]) if v in present]

print(classification_report(y_test, y_pred, target_names=target_names, labels=present))

# Get country_geomad_dem again just to show how this will be used in a train/predict command.
# country_geomad_dem = get_geomad_dem_indices(country_bounds_wgs84, stac_geoparquet, datetime_year, catalog=catalog)

stack = np.stack([country_geomad_dem[f].values.flatten() for f in feature_cols], axis=1)
stack = np.stack([country_geomad_dem[f].values.flatten() for f in feature_cols], axis=1)
stack = np.nan_to_num(stack, nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32)

predictions = model.predict(stack)

# Reshape back to raster
prediction_map = predictions.reshape(country_geomad_dem[feature_cols[0]].shape)

# Wrap in DataArray
predicted_da = xr.DataArray(
    prediction_map,
    coords={"y": country_geomad_dem.y, "x": country_geomad_dem.x},
    dims=["y", "x"],
    name="lulc",
)

In [ ]:

## Visualise our results

predicted_da = assign_crs(predicted_da, crs=analysis_crs)

# Can't use this because there is no Other (7) class in prediction.
# data_classes = sorted(colors.keys())
# print(f"All classes: {data_classes}")
data_classes = [0, 1, 2, 3, 4, 5, 6]
cmap = ListedColormap([colors[c] for c in data_classes])

predicted_da.odc.explore(
    classes=data_classes,
    cmap=cmap,
    legend=True,
    tiles=basemaps.Esri.WorldImagery
)

In [ ]:
# Aim for >80% accuracy. Don't just look at the confusion matrix, also look at the output map.

# Use a product for validation.
# One validation method for tuning and another for final measure.

# target_names = [k for k, v in sorted(classes.items(), key=lambda x: x[1]) if v != 0]
target_names = [k for k, v in sorted(classes.items(), key=lambda x: x[1]) if v != 0 and v != 7] # Use this while there is no Other (7) class in the data.

# Classification report
print(classification_report(y_test, y_pred, target_names=target_names))
print(classification_report(y_test, y_pred, target_names=target_names))

# Confusion matrix
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=target_names)
disp.plot(xticks_rotation=45, cmap="Blues")

In [ ]:
predicted_da.odc.write_cog("predicted_lulc.tif", overwrite=True)